# Mixture-of-Experts (MoE) — a toy-scale build, isolated from any one architecture

A minimal implementation of a **sparse Mixture-of-Experts feed-forward
layer** (Mixtral / Switch-Transformer style top-k routing with real sparse
dispatch and a load-balancing auxiliary loss), built as a standalone
component rather than folded into a specific attention variant — so it can
be dropped into (or compared against) any of the other architectures in
this repo.

Companion write-up: `README.md` in this folder. (K3's Stable LatentMoE, in
`../kda`, is a more elaborate relative of this same idea — latent-compressed
routed experts, a different activation, and a dense-compute simplification
rather than the real sparse dispatch built here.)

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

A normal transformer feed-forward layer is one MLP, applied to every token.
Making that MLP bigger makes the model more capable, but also makes *every
single token* pay the full cost of the bigger MLP, whether or not that
token actually needed the extra capacity.

**Mixture-of-Experts (MoE)** breaks the one big MLP into many smaller
"expert" MLPs, and routes each token to only a handful of them:

1. A small **router** network looks at each token and scores every expert.
2. Only the **top-k highest-scoring experts** actually process that token
   (`k` is usually small — 1 or 2 — even if there are dozens of experts
   total).
3. The expert outputs are combined with a weighted sum, using the router's
   own scores (renormalized over just the selected experts) as weights.

The result: the model's *total* parameter count can grow enormously (more
experts), while the *compute* per token stays roughly fixed (still only `k`
experts run per token). This is the key MoE trade: more capacity without a
proportional increase in compute per token.

## 2. The part that's easy to get wrong: load balancing

Nothing forces the router to spread tokens evenly across experts. Left
alone, it tends to **collapse**: a few experts get most of the traffic
(and therefore most of the useful gradient signal and get even better at
attracting tokens), while the rest are barely used and never improve. This
is a rich-get-richer dynamic that shows up almost immediately in practice.

The fix used here is a **load-balancing auxiliary loss**, added on top of
the normal training loss:

```
frac_tokens[e]      = fraction of tokens actually routed to expert e
mean_router_prob[e] = average router probability assigned to expert e (before top-k)
aux_loss = n_experts * sum_e( frac_tokens[e] * mean_router_prob[e] )
```

This loss is minimized when both quantities are close to uniform across
experts (`1/n_experts` each) — so the model is explicitly penalized for
letting the router concentrate on just a few experts, and gets pushed to
actually spread load out.

## 3. Real sparse dispatch (not the "compute everyone, zero the rest" shortcut)

A common shortcut — used in the `kda/` notebook's Stable LatentMoE, for
simplicity — is to run *every* expert on *every* token and multiply the
unselected ones by zero afterward. That's much simpler to write, but it
means the "sparse" layer still does dense compute under the hood.

This notebook does the real thing instead: for each expert, gather only the
tokens actually routed to it, and run the expert *only* on those tokens.
This is what a production MoE implementation does (usually with more
machinery for balancing token counts across GPUs), and it's worth seeing at
least once, since it's the part that actually delivers MoE's efficiency
promise.

> **Simplification used here:** production MoE systems also need a
> **capacity factor** — a hard cap on how many tokens any one expert can
> accept, with overflow tokens dropped or rerouted, to keep compute
> perfectly balanced across hardware. This notebook lets every selected
> token through regardless of how many other tokens picked the same
> expert, which is fine at toy scale but would cause uneven GPU utilization
> at production scale.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d, hidden_mult=4):
        super().__init__()
        h = d * hidden_mult
        self.w1 = nn.Linear(d, h, bias=False)
        self.w2 = nn.Linear(d, h, bias=False)
        self.w3 = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))   # SwiGLU expert

class SparseMoE(nn.Module):
    def __init__(self, d_model=64, n_experts=8, top_k=2, hidden_mult=4):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([Expert(d_model, hidden_mult) for _ in range(n_experts)])

    def forward(self, x):
        B, T, D = x.shape
        xf = x.reshape(-1, D)
        N = xf.shape[0]
        logits = self.router(xf)                          # N, E
        probs = logits.softmax(-1)
        topk_val, topk_idx = probs.topk(self.top_k, dim=-1)
        topk_val = topk_val / topk_val.sum(-1, keepdim=True).clamp_min(1e-9)   # renormalize over selected experts

        out = torch.zeros_like(xf)
        for e in range(self.n_experts):                     # real sparse dispatch: one pass per expert
            sel = (topk_idx == e)                             # N, top_k boolean -- did this token pick expert e?
            if not sel.any():
                continue
            rows = sel.any(dim=-1).nonzero(as_tuple=True)[0]     # which tokens actually route here
            weight = topk_val[sel]                                # matching per-token weight for this expert
            out[rows] += weight.unsqueeze(-1) * self.experts[e](xf[rows])   # run the expert ONLY on its tokens

        # load-balancing auxiliary loss (Switch-Transformer style, see README section 2)
        with torch.no_grad():
            dispatch_mask = torch.zeros(N, self.n_experts, device=x.device)
            dispatch_mask.scatter_(1, topk_idx, 1.0)
            frac_tokens = dispatch_mask.mean(0)
        mean_router_prob = probs.mean(0)
        aux_loss = self.n_experts * (frac_tokens * mean_router_prob).sum()

        return out.reshape(B, T, D), aux_loss

## 4. Assembling a tiny language model

This wraps the MoE layer in an otherwise completely ordinary transformer
block: standard causal self-attention, then the sparse MoE instead of a
plain MLP. Since `SparseMoE` returns both the layer's output *and* an
auxiliary loss, `TinyLM` collects the auxiliary losses from every layer and
adds them (lightly weighted) to the main training loss.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.scale = d_head ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)
        attn = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn = attn.masked_fill(mask, float('-inf')).softmax(-1)
        o = torch.einsum('bhts,bhsd->bhtd', attn, v).transpose(1, 2).reshape(B, T, H * Dh)
        return self.out_proj(o)

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2, aux_weight=0.01):
        super().__init__()
        self.aux_weight = aux_weight
        self.embed = nn.Embedding(vocab_size, d_model)
        self.attns = nn.ModuleList([CausalSelfAttention(d_model) for _ in range(n_layers)])
        self.moes = nn.ModuleList([SparseMoE(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        total_aux = 0.0
        for attn, moe, n1, n2 in zip(self.attns, self.moes, self.norms1, self.norms2):
            x = x + attn(n1(x))
            moe_out, aux = moe(n2(x))
            x = x + moe_out
            total_aux = total_aux + aux
        logits = self.lm_head(self.final_norm(x))
        return logits, self.aux_weight * total_aux

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through the sparse MoE layer once it's wired into a real model. So the rest of this
notebook:

1. wraps the sparse MoE layer into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

**Note:** `TinyLM` here returns `(logits, aux_loss)` rather than just
`logits` — the shared sanity/training/generation code in this repo already
handles that (it adds the auxiliary loss into the training loss
automatically when a model returns a tuple).

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Turn off the auxiliary loss** (set `aux_weight=0`) and watch expert
  usage collapse — a good way to actually *see* why the load-balancing loss
  is needed, rather than taking it on faith.
- **Log per-expert token counts** during training and plot them — you
  should see usage even out over time as the auxiliary loss does its job.
- **Add a capacity factor** — cap each expert at, say, `1.25x` the average
  expected token count per batch, and drop (or pass through unmodified) any
  overflow tokens. This is the piece that keeps compute balanced across
  hardware in a real deployment.
- **Compare against K3's Stable LatentMoE** in `../kda` — same top-k
  routing idea, but with latent-compressed routed experts and a
  dense-compute simplification instead of the real sparse dispatch built
  here.

References: Shazeer et al., *"Outrageously Large Neural Networks: The
Sparsely-Gated Mixture-of-Experts Layer,"* 2017; Fedus, Zoph, Shazeer,
*"Switch Transformers: Scaling to Trillion Parameter Models with Simple and
Efficient Sparsity,"* 2021; Jiang et al., *"Mixtral of Experts,"* 2024.